<a href="https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/12-case-studies-and-capstone/01-customer-support-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study A — Customer-support assistant, scoping to deployed (and the 2am page)

**Goal:** Thread most of the repo through *one* realistic scenario: a vague customer ask, scoped, measured, built, served, and then **debugged in production**, with runnable code at the moments that matter.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

> **How to read this.** The narrative is the case; the code cells are a *tiny working slice* of each phase (a 6-article corpus, a 5-question golden set) small enough to run in seconds but real enough to show the mechanism, especially the Phase-8 regression. The full techniques live in the sections each phase links to. This is the shape of the [capstone](CAPSTONE.md); read it before you build your own.

## Setup

Self-contained. The next two cells install the repo's `aien` helper (which pulls the `groq` client) plus `sentence-transformers` for embeddings, and load your key.

Get a free key at [console.groq.com](https://console.groq.com/); in Colab add it via the **key icon** → secret named `GROQ_API_KEY`. (Full walkthrough: [00-setup](../00-setup/00-environment.ipynb).)

In [ ]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git" sentence-transformers

In [ ]:
from aien import setup
client, MODEL = setup()   # loads GROQ_API_KEY, returns a ready Groq client + default model

## The ask (what you actually get handed)

You're an FDE at a vendor. A mid-size SaaS company, **Northwind**, signs on. The first call, their VP of Support says:

> *"Our team is drowning. Tickets are up 3x since we launched the new product and we can't hire fast enough. Can you AI something so customers get answers faster?"*

That's it. No spec, no dataset, no success metric. **This vagueness is the job**, not a failure of the customer, and the gap between this sentence and a deployed system is the entire case study.

> **🚩 Common mistake —** hearing "AI something" and immediately reaching for a framework and a vector DB. You don't yet know what "faster answers" means, what counts as *right*, or what it's worth. Build order is: **scope → measure → build → serve → operate**. Reaching for code first is how you ship the wrong thing fast.

## Phase 1 — Scope the fuzzy ask → [11 Customer craft](../11-customer-craft/01-scoping-and-discovery.ipynb)

Before any code, run the discovery questions from section 11 and turn "AI something" into a scoped, *evaluable* system.

Questions that change the design:
- **Who's the user, the customer or the support agent?** Northwind's answer: *the agent*. This is a decision, not a detail. An agent-facing "draft a reply" tool has a human in the loop (lower risk, section 07) and a different UX than a customer-facing bot. You scope the agent-assist version.
- **Answers *from what*?** Their help center (≈1,200 articles) + past resolved tickets. That's your corpus, **real and messy** (duplicates, outdated articles, tribal knowledge only in tickets).
- **What's "right"?** Not "sounds good," but *the answer a senior agent would send, grounded in a real article*. That definition becomes your eval.
- **What's it worth?** Their number: cut median handle-time, deflect the top-20 repetitive questions. That gives you a target and a budget.

The one-page **scoping doc** (section 11's artifact) comes out of this: user, problem, in/out of scope, success metric, the demo you'll show in two weeks.

> **🔵 Interview signal —** leading with "who's the user and what counts as a correct answer?" before any architecture is the senior tell. It's also what makes everything downstream measurable.

## Phase 2 — Install the metric *before* building → [02 Evals I](../02-evals-basics/01-measuring-outputs.ipynb)

Section 02's whole thesis: **put a number on "is it good" before you build the thing you'd tune.** So the next step, before retrieval and before any model call, is a tiny **golden set**: real questions paired with the answer a senior agent would send and the article it comes from.

Below is our miniature Northwind: a 6-article help center and a 5-question golden set. (A real one has ~1,200 articles and ~40 questions; the mechanics are identical.)

In [ ]:
# The corpus: a tiny help center. Each article has an id, a title, and body text.
CORPUS = {
    "acct-reset":   "Reset your password: go to Settings > Security > Reset password. A link is emailed to you.",
    "acct-email":   "Change your email: Settings > Account > Email > Edit. Confirm via the link we send.",
    "billing-refund":"Refunds are available within 30 days of purchase. Reply to your receipt email with your order id.",
    "api-status":   "Check live API status at status.northwind.io. Incidents are posted there first.",
    "api-key-err":  "Error NW-403 means an invalid API key. Regenerate the key under Settings > Developer > API keys.",
    "export-data":  "Export your data: Settings > Data > Export. You receive a CSV by email within an hour.",
}

# The golden set: question -> (the article that should answer it, the gist a senior agent would send).
GOLDEN = [
    {"q": "How do I reset my password?",        "article": "acct-reset",    "gist": "Settings > Security > Reset password; link emailed."},
    {"q": "Can I get a refund?",                 "article": "billing-refund","gist": "Within 30 days; reply to receipt with order id."},
    {"q": "I'm getting error NW-403 on the API", "article": "api-key-err",   "gist": "NW-403 = bad API key; regenerate under Settings > Developer."},
    {"q": "How do I change my email address?",   "article": "acct-email",    "gist": "Settings > Account > Email > Edit; confirm via link."},
    {"q": "How do I get my data out?",           "article": "export-data",   "gist": "Settings > Data > Export; CSV emailed within an hour."},
]
print(f"{len(CORPUS)} articles, {len(GOLDEN)} golden questions — the yardstick every later decision is measured against.")

> **⚠️ Production reality —** teams skip this because it feels like not-building. Then they "improve" the prompt for a week with no idea if it got better. The golden set is what turns that week into a graph.

## Phase 3 — Build retrieval → [03 RAG](../03-rag/00-what-is-rag.ipynb)

Now the RAG system, in the order section 03 teaches: **retrieval first** ([03/01](../03-rag/01-embeddings-retrieval.ipynb)): embed the articles, do vector search. Later you'd add hybrid + rerank ([03/02](../03-rag/02-hybrid-and-reranking.ipynb)) and revisit chunking ([03/03](../03-rag/03-chunking.ipynb)). Here we build the vector-search core and, crucially, an `index_corpus()` step we can re-run, which matters in Phase 8.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # small, CPU-friendly

# The INDEX is a separate artifact built FROM the corpus. Re-runnable on purpose:
# a RAG system's correctness depends on this index staying in sync with the corpus.
def index_corpus(corpus):
    ids = list(corpus)
    vecs = embedder.encode([corpus[i] for i in ids], normalize_embeddings=True)
    return {"ids": ids, "vecs": vecs}

def retrieve(query, index, k=1):
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    scores = index["vecs"] @ qv                 # cosine (vectors are normalized)
    top = np.argsort(scores)[::-1][:k]
    return [index["ids"][i] for i in top]

INDEX = index_corpus(CORPUS)

# Sanity check: retrieval on the golden set. Note it may miss the jargon-y NW-403 one —
# that's the vocabulary-mismatch pitfall from 03/01 that hybrid+BM25 would fix.
hits = sum(retrieve(g["q"], INDEX)[0] == g["article"] for g in GOLDEN)
print(f"retrieval hit rate: {hits}/{len(GOLDEN)}")
for g in GOLDEN:
    got = retrieve(g["q"], INDEX)[0]
    print(f"  {'OK ' if got==g['article'] else 'MISS'} {g['q'][:38]:40} -> {got}")

> **🚩 Common mistake —** treating "bad answers" as a generation problem and reaching for a bigger model. Section [03/04 (why RAG fails)](../03-rag/04-why-rag-fails.ipynb) is explicit: it's almost always **retrieval**. Your golden set tells you which: a wrong answer whose right article never got retrieved is a retrieval bug, not a model one.

## Phase 4 — Make it an assistant → [05 Agents](../05-agents/01-agent-loop-from-scratch.ipynb)

Retrieval finds articles; the assistant *drafts a reply* from them. Northwind's tickets sometimes need a lookup ("is order #4021 shipped?"), so the design gives the model a tool ([05/02](../05-agents/02-tool-design.ipynb)), but keeps a **human gate** ([05/03](../05-agents/03-guardrails-and-budgets.ipynb)): it drafts, the agent approves. That scoping decision from Phase 1 is now a safety property.

Here we keep it to the core: retrieve the article, then have the model draft a grounded reply. (The full bounded loop + `order_status` tool is section 05's job; this is the answer path the eval will grade.)

In [ ]:
def draft_reply(question, index):
    """Retrieve the most relevant article, then draft a grounded reply. Human approves before sending."""
    art_id = retrieve(question, index)[0]
    article = CORPUS[art_id]
    msgs = [
        {"role": "system", "content": "You are a support assistant. Answer ONLY from the provided article. "
                                       "Be terse. If the article doesn't cover it, say so."},
        {"role": "user", "content": f"Article:\n{article}\n\nCustomer question: {question}"},
    ]
    resp = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=120, temperature=0)
    return {"article": art_id, "reply": resp.choices[0].message.content.strip()}

demo = draft_reply("How do I reset my password?", INDEX)
print("retrieved:", demo["article"])
print("draft reply:", demo["reply"])

> **💡 Why not a bigger agent —** most of Northwind's questions are "find the article and phrase the answer." That's barely an agent. Keep the loop small; reach for a tool only where the task genuinely branches (the order lookup). Section [05/03](../05-agents/03-guardrails-and-budgets.ipynb)'s pipeline-beats-agent caution applies even inside this build. *(Case study B takes that judgment call all the way — when to drop the agent entirely.)*

## Phase 5 — The differentiator: grade answer quality → [04 Evals II](../04-evals/01-golden-sets.ipynb)

Phase 3 measured *retrieval*. Now measure **answer quality**, the thing Northwind pays for, with an **LLM-as-judge** ([04/02](../04-evals/02-llm-as-judge.ipynb)): does the drafted reply match the senior-agent gist? Wrap it into a single `eval_quality()` score so a change becomes a number, not an argument. That score, wired into CI ([04/03](../04-evals/03-regression-evals.ipynb)), is what saves you in Phase 8.

In [ ]:
import re

def judge(question, reply, gist):
    """LLM-as-judge: does the reply convey the same resolution as the senior-agent gist? 1/0."""
    msgs = [
        {"role": "system", "content": "You grade support replies. Output ONLY '1' if the reply conveys the "
                                       "same core resolution as the reference, else '0'. No other text."},
        {"role": "user", "content": f"Question: {question}\nReference: {gist}\nReply: {reply}\nGrade (1 or 0):"},
    ]
    out = client.chat.completions.create(model=MODEL, messages=msgs, max_tokens=2, temperature=0)
    return 1 if "1" in (out.choices[0].message.content or "") else 0

def eval_quality(index, verbose=False):
    """The whole system, end to end, over the golden set -> one quality score in [0,1]."""
    passed = 0
    for g in GOLDEN:
        d = draft_reply(g["q"], index)
        ok = judge(g["q"], d["reply"], g["gist"])
        passed += ok
        if verbose:
            print(f"  {'PASS' if ok else 'FAIL'} [{d['article']}] {g['q'][:36]}")
    return passed / len(GOLDEN)

baseline = eval_quality(INDEX, verbose=True)
print(f"\nBASELINE quality score: {baseline:.0%}")

> **🔵 Interview signal —** "we had an LLM-judge eval gate in CI, spot-checked against human labels" is the line that separates an engineer who *built a demo* from one who *shipped a system*. It's the honest answer to "how did you know it was good?"

## Phase 6 — Serve & size → [09 Serving](../09-serving-inference/01-serving-frameworks.ipynb) + [10 System design](../10-ml-system-design/01-designing-an-inference-service.ipynb)

Northwind has ~200 agents, bursty in business hours. Size it with the section-10 method. No new code, just the arithmetic an interviewer expects:

- **Managed endpoint or self-host?** At this volume a **managed API** (the OpenAI-compatible seam this repo uses) is cheaper and less work than standing up GPUs, and it's [09/01](../09-serving-inference/01-serving-frameworks.ipynb)'s default. Note the self-host-on-vLLM path for when volume justifies it.
- **The numbers** ([10/01](../10-ml-system-design/01-designing-an-inference-service.ipynb)): peak concurrent agents → QPS → tokens/s → cost/month, with a prompt cache for the repeated system preamble.

In [ ]:
# The back-of-envelope from 10/01 — the answer to "how many GPUs / what's the bill?"
def size_it(peak_concurrent_agents, reqs_per_agent_per_min, tokens_per_req, tok_per_s_per_gpu=2500, util=0.7):
    import math
    qps = peak_concurrent_agents * reqs_per_agent_per_min / 60
    demand_tok_s = qps * tokens_per_req
    gpus = math.ceil(demand_tok_s / (tok_per_s_per_gpu * util))
    return {"qps": round(qps, 1), "demand_tok_s": round(demand_tok_s), "gpus_if_self_hosted": gpus}

print(size_it(peak_concurrent_agents=200, reqs_per_agent_per_min=2, tokens_per_req=300))
print("-> at this scale, a managed endpoint is the call; self-host on vLLM when the bill crosses GPU break-even.")

> **⚠️ Production reality —** the interviewer will push "what at 10x?" Because you sized it, you answer with a lever, not a redesign: more replicas, or self-host on vLLM when the managed bill crosses the GPU break-even.

## Phase 7 — Ship the demo → back to [11 Customer craft](../11-customer-craft/01-scoping-and-discovery.ipynb)

Two weeks in, you demo to Northwind **on their real tickets, live**, not a cherry-picked script (section 11's demo discipline). You show the eval number, not vibes: *"on the golden set, N% matched a senior-agent reply, grounded in a real article; here are the ones it got wrong and why."* Honest failure cases build more trust than a flawless scripted run.

You wire in **observability** ([08/01](../08-operations/01-observability-and-llmops.ipynb)) at launch: trace every call, log prompts safely (no customer PII, which ties to [07](../07-security/01-prompt-injection-and-trust.ipynb)), track cost/latency/error rate. It goes live to a pilot group.

## Phase 8 — Two weeks later, the 2am page (build → **debug**)

The system was working. Now pilot agents report: **"the answers went weird."** Quality is down, nobody deployed a model change, and the eval gate in CI is **green**. This is the other half of the job, **diagnosing a live regression**, and it's where the observability and evals you built earn their cost.

We can *reproduce this for real*. Northwind's content team did a help-center cleanup and **rewrote and re-titled articles**, but nobody re-ran `index_corpus()`. The index now points at the old text. Watch the score drop, then recover.

In [ ]:
# The content team's migration: articles are rewritten/expanded (same ids, new wording & structure).
CORPUS_V2 = dict(CORPUS)
CORPUS_V2["acct-reset"]  = ("Forgot or resetting credentials? Navigate to your profile, open the Security "
                            "panel, and choose 'Reset password'. We dispatch a one-time link to your inbox.")
CORPUS_V2["api-key-err"] = ("Authentication failures (formerly NW-403) are now surfaced as AUTH_INVALID_KEY. "
                            "Rotate credentials from the Developer console under API credentials.")
CORPUS_V2["billing-refund"] = ("Our refund window is one calendar month from the transaction date. Initiate "
                               "from the Billing portal, or reply to the emailed receipt with your reference id.")

# THE BUG: the corpus changed, but the index was NOT rebuilt. Retrieval still scores against old vectors.
CORPUS.update(CORPUS_V2)          # production content is now V2...
# ...but INDEX is stale (still the V1 vectors from Phase 3).

regressed = eval_quality(INDEX, verbose=True)
print(f"\nPRODUCTION quality score now: {regressed:.0%}   (baseline was {baseline:.0%})")
print("CI is green because CI runs against a FROZEN copy — it never saw the migration.")

In [ ]:
# The fix: re-index the migrated corpus. The seam we built in Phase 3 is a one-liner to re-run.
INDEX = index_corpus(CORPUS)      # rebuild vectors from the current (V2) content
recovered = eval_quality(INDEX)
print(f"after re-indexing: {recovered:.0%}   (was {regressed:.0%} while stale)")
print("\nDurable fix: add index_corpus() to the content-publishing workflow, plus a daily CANARY eval")
print("that runs the golden set against PRODUCTION — so the next drift pages a dashboard, not an agent.")

**The diagnosis, as a reusable decision tree** (this is the skill, not the specific bug):

1. **Reproduce with a number, not a vibe.** Run the [04 regression eval](../04-evals/03-regression-evals.ipynb) against *production*, not the frozen CI copy. The score really dropped.
2. **Retrieval or generation?** ([03/04 lens](../03-rag/04-why-rag-fails.ipynb)) Pull traces ([08/01](../08-operations/01-observability-and-llmops.ipynb)): the model's replies are fine *given* what it retrieved, but retrieval returns near-misses. **It's retrieval.**
3. **What changed, if not the code?** The **data** moved: articles were rewritten upstream. The index is **stale**.
4. **Root cause:** nobody re-indexed after the migration, a **data-freshness dependency** no one owned.

> **🚩 Common mistake —** assuming a quality regression means "the model got worse" or "someone changed the prompt." In a RAG system the most common cause is **the data moved and the index didn't** — a green CI gate can't catch drift it never sees. Building it is wiring the index; debugging it is knowing to suspect the data boundary first.

> **🔵 Interview signal —** "our CI eval was green but production quality dropped, because the corpus was re-indexed upstream and our vectors went stale — so we added a production canary eval" is a *senior* war story. It shows you separate offline eval from production monitoring, and think in data dependencies, not just code.

## What this case exercised

One scenario, most of the repo, and both halves of the job:

| Phase | Section | Skill |
|---|---|---|
| Scope the ask | [11](../11-customer-craft/01-scoping-and-discovery.ipynb) | discovery, the scoping doc |
| Metric first | [02](../02-evals-basics/01-measuring-outputs.ipynb) | golden set before building |
| Retrieval | [03](../03-rag/00-what-is-rag.ipynb) | RAG, hybrid, chunking, failure diagnosis |
| Assistant | [05](../05-agents/01-agent-loop-from-scratch.ipynb) | agent loop, tools, guardrails |
| Quality gate | [04](../04-evals/01-golden-sets.ipynb) | LLM-judge, regression-as-CI |
| Serve & size | [09](../09-serving-inference/01-serving-frameworks.ipynb), [10](../10-ml-system-design/01-designing-an-inference-service.ipynb) | managed vs self-host, sizing math |
| Security | [07](../07-security/01-prompt-injection-and-trust.ipynb) | untrusted tickets, safe logging, human gate |
| Launch + demo | [11](../11-customer-craft/01-scoping-and-discovery.ipynb), [08](../08-operations/01-observability-and-llmops.ipynb) | honest demo, tracing |
| **The 2am page** | [08](../08-operations/01-observability-and-llmops.ipynb), [04](../04-evals/03-regression-evals.ipynb), [03/04](../03-rag/04-why-rag-fails.ipynb) | **diagnosing a live regression** |

> **⭐ Key takeaway —** you were never missing the pieces; you built them across sections 00–11. A real project is where you *compose* them under a customer's constraints, ship honestly, and — the half most portfolios skip — **keep it working after launch.** That arc, told clearly, is the strongest thing you can bring to an interview. Now go build your own: the [capstone](CAPSTONE.md).